In [0]:
%sql
SELECT CURRENT_METASTORE()



In [0]:
%sql
CREATE TABLE dbacademy.default.machine_telemetry_test (
    machine_id STRING,
    event_time TIMESTAMP,
    temperature DOUBLE,
    vibration DOUBLE,
    rpm INT
  );

In [0]:
%sql
INSERT INTO dbacademy.default.machine_telemetry_test
VALUES (
  'MACHINE-01',
  current_timestamp(),
  82.5,
  3.8,
  1450
);

SELECT *
FROM dbacademy.default.machine_telemetry_test
ORDER BY event_time DESC
LIMIT 5;

In [0]:
%sql
DESCRIBE TABLE dbacademy.default.machine_telemetry_test;


In [0]:
%sql
GRANT USE CATALOG
ON CATALOG dbacademy
TO `b1f77037-f0f5-4fb2-878e-97070957119d`;

In [0]:
%sql
GRANT USE SCHEMA
ON SCHEMA dbacademy.default
TO `b1f77037-f0f5-4fb2-878e-97070957119d`;

In [0]:
%sql
GRANT SELECT, MODIFY
ON TABLE dbacademy.default.machine_telemetry_test
TO `b1f77037-f0f5-4fb2-878e-97070957119d`

In [0]:
%sql
SHOW GRANTS ON TABLE dbacademy.default.machine_telemetry_test;

In [0]:
%sql
DESCRIBE DETAIL dbacademy.default.machine_telemetry_test;

In [0]:
%sql
SHOW GRANTS ON TABLE dbacademy.default.machine_telemetry_test;

In [0]:
%sql
DESCRIBE TABLE EXTENDED dbacademy.default.machine_telemetry_test;

In [0]:
%sql
SELECT
    machine_id,
    event_time,
    temperature,
    vibration,
    rpm
FROM dbacademy.default.machine_telemetry_test
WHERE machine_id = 'ZEROBUS-TEST-01'
ORDER BY event_time DESC
LIMIT 5;

In [0]:
%sql
SELECT
    machine_id,
    COUNT(*) AS readings,
    ROUND(AVG(temperature), 2) AS avg_temperature,
    ROUND(MAX(temperature), 2) AS max_temperature,
    ROUND(AVG(vibration), 2) AS avg_vibration,
    ROUND(AVG(rpm), 0) AS avg_rpm,
    MAX(event_time) AS latest_reading
FROM dbacademy.default.machine_telemetry_test
WHERE machine_id IN (
    'MACHINE-01',
    'MACHINE-02',
    'MACHINE-03'
)
GROUP BY machine_id
ORDER BY machine_id;

In [0]:
%sql
SELECT
    machine_id,
    event_time,
    temperature,
    vibration,
    rpm
FROM dbacademy.default.machine_telemetry_test
WHERE machine_id IN (
    'MACHINE-01',
    'MACHINE-02',
    'MACHINE-03'
)
ORDER BY event_time DESC
LIMIT 30;

In [0]:
%sql

CREATE OR REPLACE VIEW dbacademy.default.machine_health_current AS
WITH ranked_readings AS (
    SELECT
        machine_id,
        event_time,
        temperature,
        vibration,
        rpm,
        ROW_NUMBER() OVER (
            PARTITION BY machine_id
            ORDER BY event_time DESC
        ) AS rn
    FROM dbacademy.default.machine_telemetry_test
)
SELECT
    machine_id,
    event_time,
    temperature,
    vibration,
    rpm,
    CASE
        WHEN temperature >= 80
          OR vibration >= 2.5
          OR rpm < 1000
        THEN 'WARNING'
        ELSE 'NORMAL'
    END AS machine_status,
    CASE
        WHEN temperature >= 80 THEN 'HIGH TEMPERATURE'
        WHEN vibration >= 2.5 THEN 'HIGH VIBRATION'
        WHEN rpm < 1000 THEN 'LOW RPM'
        ELSE 'ALL READINGS WITHIN PROTOTYPE THRESHOLDS'
    END AS status_reason
FROM ranked_readings
WHERE rn = 1;


In [0]:
%sql

SELECT
    machine_id,
    event_time,
    temperature,
    vibration,
    rpm,
    machine_status,
    status_reason
FROM dbacademy.default.machine_health_current
ORDER BY machine_id;


In [0]:
%sql

INSERT INTO dbacademy.default.machine_telemetry_test
    (machine_id, event_time, temperature, vibration, rpm)
VALUES
    (
        'MACHINE-01',
        current_timestamp(),
        85.0,
        1.4,
        1430
    );


In [0]:
%sql

SELECT
    machine_id,
    event_time,
    temperature,
    vibration,
    rpm,
    machine_status,
    status_reason
FROM dbacademy.default.machine_health_current
WHERE machine_id = 'MACHINE-01';


In [0]:
%sql
--Step 1 — Create a dashboard summary query

SELECT
    machine_id,
    event_time,
    temperature,
    vibration,
    rpm,
    machine_status,
    status_reason,
    CASE
        WHEN machine_status = 'WARNING' THEN 1
        ELSE 0
    END AS warning_flag
FROM dbacademy.default.machine_health_current
WHERE machine_id IN (
    'MACHINE-01',
    'MACHINE-02',
    'MACHINE-03'
)
ORDER BY machine_id;
